In [ ]:
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys

In [ ]:
## find CASSIS LTE PYTHON package here: https://gitlab.in2p3.fr/sandrine.bottinelli/cassis-lte-python
## this is used to collect molecular data
sys.path.append("/Users/pamela/Documents/phd/cassis-lte-python-master/")
#%run /Users/pamela/Documents/phd/baseline_als.py

In [ ]:
from cassis_lte_python.database.transitions import get_transition_list
from cassis_lte_python.database.species import get_partition_function
from cassis_lte_python.database.species import get_species_info
from cassis_lte_python.database.setupdb import DATABASE_SQL
db = DATABASE_SQL

Using database : /Users/Pamela/Applications/CASSIS/database/CASSIS20240415+LSD+HFS+NIST.db


In [ ]:
## get list of transitions and their parameters using cassis_lte_python get_transition_list
## ENTER VALUES HERE
mol_tag_value = 31801
mol_name = "CH3NH2" ## species name, note the name coming from the list may have additional characters or formatting

## do not need to enter values here
mol_tag = "0"+str(mol_tag_value)
molecule = mol_tag+" "+mol_name
transitions = get_transition_list(mol_tag, fmhz_ranges=([1000, 200000000.]), database=db, return_type='df', **{mol_tag:{'eup_max': 20000., 'aij_min':1.0e-20}})
#transitions

## make dataframe with information needed for the linelist file
df_transitions = pd.DataFrame({'id': np.arange(0,len(transitions),1), 'fMHz': transitions['fMHz'], 'aud': transitions['aij'], 'elo': transitions['elow'], 'eup': transitions['eup'], 'glo': np.append(1,transitions['igu'][0:-1]), 'gup': transitions['igu']})
df_transitions

,id,fMHz,aud,elo,eup,glo,gup
0,0,1.019638e+03,1.666821e-11,972.998,1399.976423,1,219
1,1,1.032863e+03,1.753558e-11,1138.357,1637.891842,219,79
2,2,1.096042e+03,1.760187e-14,1975.840,2842.846449,79,101
3,3,1.096300e+03,2.111353e-14,1161.420,1671.077408,101,231
4,4,1.100440e+03,2.134702e-14,1975.773,2842.750262,231,303
...,...,...,...,...,...,...,...
46207,46207,2.998214e+06,1.216514e-06,475.564,828.122350,213,147
46208,46208,2.998297e+06,2.958374e-07,475.708,828.333516,147,49
46209,46209,2.998523e+06,8.337491e-08,1333.354,2062.305981,49,201
46210,46210,2.999591e+06,2.952172e-03,1215.198,1892.357057,201,83


In [ ]:
## get partition function information using cassis_lte_python get_partition_function, make dataframe with that info
part_func = get_partition_function(mol_tag_value)
temp = part_func[0][::-1]
part = part_func[1][::-1]
df_partition = pd.DataFrame({'temp': temp, 'part': part})
print(df_partition)

        temp    part
0   1000.000  5.2789
1    500.000  4.8269
2    300.000  4.4939
3    225.000  4.3063
4    150.000  4.0419
5     75.000  3.5900
6     37.500  3.1380
7     18.750  2.6861
8      9.375  2.2342
9      5.000  1.8243
10     2.725  1.4286


In [ ]:
## make file
filename="linelist/linelist_"+mol_name+".inp"
if os.path.exists(filename):
    os.remove(filename)

In [ ]:
## write file
file_obj = open(filename, mode="a")
file_obj.write("! RADMC-3D Standard line list -- Created using the CDMS database (Müller et al 2001, 2005)\n")
file_obj.write("! Format number:\n 1 \n")
file_obj.write("! Molecule name:\n"+mol_name+"\n")
file_obj.write("! Reference: CDMS  -- #"+mol_tag+"\n")
file_obj.write("! Molecular weight (in atomic units) \n"+str(get_species_info(mol_tag_value)['molecular_mass'])+"\n")
file_obj.write("! Include table of partition sum? (0=no, 1=yes) \n"+"1"+"\n")
file_obj.write("! Include additional information? (0=no, 1=yes) \n"+"0"+"\n")
file_obj.write("! Nr of temperature points for the partition sum \n"+str(len(part))+"\n")
file_obj.write("!  Temp [K]      PartSum \n")
for row in range(0,len(df_partition),1):
    file_obj.write(str(f"{np.array(df_partition['temp'][row]):.6E}")+" "+str(f"{10**np.array(df_partition['part'][row]):.6E}")+"\n")
#df.to_csv('table_test.dat', mode='a', sep='\t', header=False, index=False)
file_obj.write("! Nr of lines \n"+str(len(df_transitions))+"\n")
file_obj.write("! ID    Lambda [mic]  Aud [sec^-1]  E_lo [cm^-1]  E_up [cm^-1]  g_lo  g_up \n")
for row in range(0,len(df_transitions),1):
    file_obj.write(str(df_transitions['id'][row]+1)+" "+str(f"{(df_transitions['fMHz'][row]*u.MHz).to(u.um, equivalencies=u.spectral()).value:.6E}")+" "+str(f"{df_transitions['aud'][row]:.6E}")+" "+str(f"{df_transitions['elo'][row]:.6E}")+" "+str(f"{df_transitions['eup'][row]/1.43877:.6E}")+" "+str(f"{df_transitions['glo'][row]:.6E}")+" "+str(f"{df_transitions['gup'][row]:.6E}")+"\n")


In [ ]:
file_obj.close()